In [38]:
!pip install faker

In [39]:
import pandas as pd
import random
import sqlite3
from faker import Faker
from datetime import datetime


In [40]:
fake = Faker()

In [41]:
# CONFIGURATION
NUM_PATIENTS = 1200
NUM_DOCTORS = 60
NUM_APPOINTMENTS = 3500
NUM_PRESCRIPTIONS = 3200

In [42]:
# 1. Generate Patients Table
patients = []

for i in range(1, NUM_PATIENTS + 1):

    dob = fake.date_of_birth(minimum_age=0, maximum_age=90)

    age = datetime.now().year - dob.year

    gender = random.choice(["Male", "Female", "Other", "M", "F"])  # inconsistent values

    # introduce missing blood group
    blood_group = random.choice(["A+", "A-", "B+", "B-", "O+", "AB+", None])

    phone = fake.phone_number()

    patients.append({
        "patient_id": i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "gender": gender,                         # Nominal
        "date_of_birth": dob,
        "age": age,                               # Ratio
        "admission_year": random.randint(2015, 2025),  # Interval
        "blood_group": blood_group,
        "phone": phone
    })

patients_df = pd.DataFrame(patients)

duplicates = patients_df.sample(10).copy()
duplicates["patient_id"] = range(NUM_PATIENTS + 1, NUM_PATIENTS + 11)

patients_df = pd.concat([patients_df, duplicates], ignore_index=True)

patients_df.to_csv("patients.csv", index=False)


In [43]:
# 2. Generate Doctors Table

specializations = [
    "Cardiology", "Neurology", "Orthopedics",
    "Pediatrics", "Dermatology", "General Medicine"
]

doctors = []

for i in range(1, NUM_DOCTORS + 1):

    experience = random.randint(1, 40)

    salary = random.randint(40000, 150000)

    # introduce salary outliers
    if random.random() < 0.02:
        salary = salary * 5

    doctors.append({
        "doctor_id": i,
        "name": fake.name(),
        "specialization": random.choice(specializations),   # Nominal
        "experience_years": experience,                     # Ratio
        "salary": salary                                    # Ratio
    })

doctors_df = pd.DataFrame(doctors)

# introduce missing specialization
doctors_df.loc[random.sample(range(NUM_DOCTORS), 3), "specialization"] = None

doctors_df.to_csv("doctors.csv", index=False)


In [44]:

# 3. Generate Appointments Table

severity_levels = ["Mild", "Moderate", "Severe", "Critical"]  # Ordinal

appointments = []

for i in range(1, NUM_APPOINTMENTS + 1):

    cost = round(random.uniform(50, 2000), 2)

    # introduce cost outliers
    if random.random() < 0.02:
        cost = cost * 10

    satisfaction = random.randint(1, 10)

    # introduce missing satisfaction
    if random.random() < 0.05:
        satisfaction = None

    appointments.append({
        "appointment_id": i,
        "patient_id": random.randint(1, NUM_PATIENTS),
        "doctor_id": random.randint(1, NUM_DOCTORS),
        "appointment_date": fake.date_between(start_date='-3y', end_date='today'),
        "severity_level": random.choice(severity_levels),   # Ordinal
        "treatment_cost": cost,                             # Ratio
        "satisfaction_score": satisfaction                  # Interval
    })

appointments_df = pd.DataFrame(appointments)

appointments_df.to_csv("appointments.csv", index=False)


In [45]:
# 4. Generate Prescriptions Table

medications = [
    "Paracetamol", "Ibuprofen", "Amoxicillin",
    "Metformin", "Atorvastatin", "Omeprazole",
    "Aspirin", "Azithromycin"
]

prescriptions = []

for i in range(1, NUM_PRESCRIPTIONS + 1):

    dosage = random.randint(10, 500)

    duration = random.randint(1, 30)

    # introduce unrealistic values
    if random.random() < 0.01:
        dosage = 2000

    prescriptions.append({
        "prescription_id": i,
        "appointment_id": random.randint(1, NUM_APPOINTMENTS),
        "medication_name": random.choice(medications),
        "dosage_mg": dosage,              # Ratio
        "duration_days": duration         # Ratio
    })

prescriptions_df = pd.DataFrame(prescriptions)

prescriptions_df.to_csv("prescriptions.csv", index=False)

print("CSV files generated.")

CSV files generated.


In [46]:
# 5. CREATE SQLITE DATABASE
conn = sqlite3.connect("healthcare.db")

cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

cursor.executescript("""

DROP TABLE IF EXISTS prescriptions;
DROP TABLE IF EXISTS appointments;
DROP TABLE IF EXISTS doctors;
DROP TABLE IF EXISTS patients;

CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    gender TEXT,
    date_of_birth DATE,
    age INTEGER,
    admission_year INTEGER,
    blood_group TEXT,
    phone TEXT
);

CREATE TABLE doctors (
    doctor_id INTEGER PRIMARY KEY,
    name TEXT,
    specialization TEXT,
    experience_years INTEGER,
    salary REAL
);

CREATE TABLE appointments (
    appointment_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    doctor_id INTEGER,
    appointment_date DATE,
    severity_level TEXT,
    treatment_cost REAL,
    satisfaction_score INTEGER,
    FOREIGN KEY(patient_id) REFERENCES patients(patient_id),
    FOREIGN KEY(doctor_id) REFERENCES doctors(doctor_id)
);

CREATE TABLE prescriptions (
    prescription_id INTEGER PRIMARY KEY,
    appointment_id INTEGER,
    medication_name TEXT,
    dosage_mg INTEGER,
    duration_days INTEGER,
    FOREIGN KEY(appointment_id) REFERENCES appointments(appointment_id)
);

""")

# Insert data into tables

patients_df.to_sql("patients", conn, if_exists="append", index=False)
doctors_df.to_sql("doctors", conn, if_exists="append", index=False)
appointments_df.to_sql("appointments", conn, if_exists="append", index=False)
prescriptions_df.to_sql("prescriptions", conn, if_exists="append", index=False)

conn.commit()
conn.close()

print("SQLite database 'healthcare.db' created successfully!")

SQLite database 'healthcare.db' created successfully!
